---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

*This notebook is the assignment template. Each group works on its own copy, and cells or sections may be added as needed.*

---

## Group 16

- Audric (13524010)
- Mahatma (13524015)
- Ishak (13524022)
- Nathanael (13524066)
- Richard (13524112)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

Cristiano Ronaldo sedang mengembangkan bisnisnya ke bidang hospitalitas dan ingin membuat sebuah penginapan, Ronaldo Hotel. Ia meminta bantuan kepada asisten Lab AI STEI ITB  untuk membantunya membuat sistem cerdas untuk dapat mengestimasi pengunjung yang akan membatalkan pesanannya agar penginapannya dapat memiliki gambaran yang lebih akurat. Sistem tersebut menggunakan pembelajaran mesin klasik untuk mendapatkan kinerja yang baik tanpa sumber daya komputasi yang besar seperti Deep Learning. Karena para asisten sedang sibuk, tugas ini diserahkan kepada peserta kuliah sebagai anggota akademi La Fabrica.



## Overview

Data yang diberikan berisi riwayat pemesanan penginapan beserta status batal atau tidaknya, dengan fitur numerik dan kategorikal. Sebelum dipakai, data diolah melalui tahap data cleaning, data transformation, feature selection, dan dimensionality reduction. Model kemudian dilatih dan divalidasi.
Algoritma yang akan diimplementasikan from scratch adalah KNN, Decision Tree Learning (DTL), Logistic Regression, SVM, dan Bayesian Network (Naive Bayes). Hasil setiap algoritma dibandingkan dengan implementasi pustaka scikit-learn, dan prediksi pada data uji dikumpulkan ke kompetisi Kaggle.


## Aim

1. Membersihkan dan menyiapkan data dengan alasan yang jelas di setiap langkah, dengan data latih dan data uji diproses secara terpisah.
2. Mengimplementasikan kelima algoritma sendiri, hanya dengan bantuan pustaka perhitungan seperti NumPy.
3. Membandingkan setiap model *from scratch* dengan versi scikit-learn memakai metrik yang sama.
4. Menghasilkan berkas submisi Kaggle yang bisa dibuat ulang dari notebook ini.

## Metric

Kompetisi memakai Macro F1-score, yaitu rata-rata F1-score kelas batal (1) dan kelas tidak batal (0)

## Dataset

Data latih berisikan 27.373 pesanan dengan 19 kolom, yaitu id, 17 fitur, dan jawaban is_canceled. Data uji berisi 6.844 pesanan dengan kolom yang sama, tetapi tanpa jawaban. Dari seluruh data latih, 8.969 atau sekitar 32,8% pesanan batal. Artinya, hanya satu dari tiga pesanan yang batal.


## Approach

1. Data latih dibagi menjadi data latih (80%) dan data validasi (20%) secara stratified sebelum pengolahan apa pun.
2. Cleaning, encoding, dan scaling di-fit hanya pada data latih, lalu dipakai ulang untuk data validasi dan data uji melalui `runPreprocessing()`.
3. Setiap model dilatih pada data latih dan dinilai pada data validasi.
4. Model disimpan, dimuat ulang, lalu model final dilatih ulang pada seluruh `train.csv` untuk submisi Kaggle.

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


## Import Libraries

Every import of the notebook is kept in this cell.

In [ ]:
import os
import random
import calendar
import math
import pickle
import subprocess
from collections import Counter
from collections.abc import Hashable, Mapping, Sequence
from dataclasses import dataclass
from functools import partial
from numbers import Integral, Real
from pathlib import Path
from tempfile import NamedTemporaryFile
from textwrap import fill

import numpy as np
import pandas as pd
from matplotlib.figure import Figure
from IPython.display import Image, display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB



## Settings

Paths and shared constants are kept in this class.

In [ ]:
class Settings:
    ID_COL = "id"
    TARGET_COL = "is_canceled"
    VAL_SIZE = 0.2
    SEED       = 42
    DATA_DIR   = Path('data')
    OUTPUT_DIR = Path('output')
    MODELS_DIR = OUTPUT_DIR / 'models'
    MEDIA_DIR  = OUTPUT_DIR / 'media'

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ID_COL, TARGET_COL = CFG.ID_COL, CFG.TARGET_COL
VAL_SIZE, SEED = CFG.VAL_SIZE, CFG.SEED
MODELS_DIR, MEDIA_DIR = CFG.MODELS_DIR, CFG.MEDIA_DIR

OUTPUT_DIR = CFG.OUTPUT_DIR
FEATURE_COUNT = None
PCA_COMPONENTS = None
OVERSAMPLE = False

NOTEBOOK_CONFIG = {
    "knn_candidates": tuple(range(2, 34)),
    "dtl": {
    "max_depth": 14,
    "min_samples_leaf": 5,
    "min_samples_split": 2,
    "min_impurity_decrease": 0.0,
},
    "logistic_regression": {
        "regularization": "none",
        "epochs": 100,
        "randomState": SEED,
    },
    "svm": {
        "C": 10,
        "kernel": "rbf",
        "gamma": 0.2,
        "max_iter": 800_000,
        "random_state": SEED,
    },
    "naive_bayes": {"smoothing": 0.1, "pick_features": True},
    "svm_train_limit": 3000,
    "submission_model": "svm",
}

SVM_TRAIN_LIMIT = NOTEBOOK_CONFIG["svm_train_limit"]
SUBMISSION_MODEL = NOTEBOOK_CONFIG["submission_model"]

## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
CFG.DATA_DIR = '/kaggle/input/competitions/T2-IF3170-2026-2027/'
train_df = pd.read_csv(CFG.DATA_DIR + 'train.csv', dtype={'id': str})
test_df = pd.read_csv(CFG.DATA_DIR + 'test.csv', dtype={'id': str})
template = pd.read_csv(CFG.DATA_DIR + 'sample_submission.csv', dtype={'id': str})
template = pd.read_csv(
    CFG.DATA_DIR + "sample_submission.csv",
    dtype={ID_COL: str},
)

print(f'Train: {train_df.shape}  Test: {test_df.shape}')

Train: (27373, 19)  Test: (6844, 18)


---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [ ]:
display(train_df.head())
display(train_df.dtypes.rename("tipe_data").to_frame())

,id,num_adults,num_children,weekend_nights,weekday_nights,meal_package,needs_parking,room_category,days_before_arrival,arrival_year,arrival_month,arrival_day,booking_channel,is_returning_guest,prior_cancellations,prior_completed_bookings,nightly_rate_idr,special_requests,is_canceled
0,57152f82,2.0,NaN,1,2,Breakfast,0.0,Type B,88.0,2024,4,3,Offline Agent,0,0,0,1330000.0,0.0,1
1,af2ec71b,2.0,0.0,2,1,Half Board,0.0,Type D,120.0,2023,8,1,Offline Agent,0,0,0,NaN,0.0,0
2,5d5469ab,3.0,0.0,1,2,Breakfast,0.0,Type G,209.0,2024,8,28,Online Travel Agent,0,0,0,2099000.0,0.0,1
3,d2df0369,2.0,1.0,2,1,Breakfast,0.0,Type G,93.0,2024,8,13,Online Travel Agent,0,0,0,1693000.0,2.0,0
4,91d253db,2.0,0.0,0,1,Room Only,0.0,Type B,14.0,2024,9,1,Online Travel Agent,0,0,0,2010000.0,2.0,0


,tipe_data
id,object
num_adults,float64
num_children,float64
weekend_nights,int64
weekday_nights,int64
meal_package,object
needs_parking,float64
room_category,object
days_before_arrival,float64
arrival_year,int64


## Explore Data

This part covers distributions and relationships between attributes.

In [ ]:
display(train_df[TARGET_COL].value_counts().sort_index().rename("jumlah").to_frame())

,jumlah
is_canceled,
0,18404
1,8969


## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
# Write your code here
input_features = train_df.columns.drop([ID_COL, TARGET_COL])
display(pd.DataFrame({
    "missing_train": train_df[input_features].isna().sum(),
    "missing_test": test_df[input_features].isna().sum(),
}))

,missing_train,missing_test
num_adults,809,180
num_children,3035,747
weekend_nights,0,0
weekday_nights,0,0
meal_package,1760,442
needs_parking,1082,247
room_category,1002,242
days_before_arrival,1123,290
arrival_year,0,0
arrival_month,0,0


---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
def split_data(train, test):
    X = train.drop(columns=[ID_COL, TARGET_COL]).copy()
    y = train[TARGET_COL].copy()

    X_test = test.drop(columns=[ID_COL]).copy()
    id_test = test[ID_COL].copy()

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=VAL_SIZE, stratify=y, random_state=SEED
    )
    keep = ~X_train.duplicated()
    X_train, y_train = X_train[keep], y_train[keep]

    return X_train, X_val, y_train, y_val, X_test, id_test

In [ ]:
X_train, X_val, y_train, y_val, X_test, id_test = split_data(train_df, test_df)
print("Train", X_train.shape)
print("Validasi", X_val.shape)
print("Test", X_test.shape)

Train (21857, 17)
Validasi (5475, 17)
Test (6844, 17)


---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

split_data() memisahkan is_canceled sebagai target dan id sebagai pengenal. Model memakai 17 kolom lainnya sebagai fitur awal.

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

In [ ]:
MIN_GROUP_ROWS = 30

MODE_COLS = [
    "num_adults",
    "num_children",
    "special_requests",
    "needs_parking",
    "meal_package",
    "room_category",
    "booking_channel",
    "weekend_nights",
    "weekday_nights",
    "arrival_year",
    "arrival_month",
    "arrival_day",
    "is_returning_guest",
    "prior_cancellations",
    "prior_completed_bookings",
]

MEDIAN_COLS = [
    "days_before_arrival",
]


def get_mode(data, col):
    modes = data[col].mode(dropna=True)
    if len(modes) == 0:
        raise ValueError(f"Kolom '{col}' kosong semua di train split; gak bisa diisi.")
    value = modes.iloc[0]
    if isinstance(value, str):
        return value
    return float(value)


def get_median(data, col):
    med = data[col].median()
    if np.isnan(med):
        raise ValueError(f"Kolom '{col}' kosong semua di train split; gak bisa diisi.")
    return float(med)


def room_month_price_med(data):
    has_price = data["nightly_rate_idr"].notna()
    has_room = data["room_category"].notna()
    has_month = data["arrival_month"].notna()
    complete = data.loc[has_price & has_room & has_month]

    groups = complete.groupby(["room_category", "arrival_month"])["nightly_rate_idr"]
    n_rows = groups.count()
    med = groups.median()

    result = {}
    for (room, month), count in n_rows.items():
        if count >= MIN_GROUP_ROWS:
            result[(room, month)] = float(med[(room, month)])
    return result


def room_price_med(data):
    has_price = data["nightly_rate_idr"].notna()
    has_room = data["room_category"].notna()
    complete = data.loc[has_price & has_room]

    med = complete.groupby("room_category")["nightly_rate_idr"].median()

    result = {}
    for room, value in med.items():
        result[room] = float(value)
    return result


def month_price_med(data):
    has_price = data["nightly_rate_idr"].notna()
    has_month = data["arrival_month"].notna()
    complete = data.loc[has_price & has_month]

    med = complete.groupby("arrival_month")["nightly_rate_idr"].median()

    result = {}
    for month, value in med.items():
        result[month] = float(value)
    return result


def fill_col(data, col, value, how, logs):
    is_nan = data[col].isna()
    data.loc[is_nan, col] = value
    log_step(logs, 5, f"NaN diisi {how}", col, is_nan.sum())


class MissingValueImputer:
    def __init__(self):
        self.fill_values_ = None
        self.room_month_med_ = None
        self.room_med_ = None
        self.month_med_ = None
        self.global_med_ = None

    def fit(self, data):
        fill_values = {}
        for col in MODE_COLS:
            fill_values[col] = get_mode(data, col)
        for col in MEDIAN_COLS:
            fill_values[col] = float(np.round(get_median(data, col)))

        self.fill_values_ = fill_values
        self.room_month_med_ = room_month_price_med(data)
        self.room_med_ = room_price_med(data)
        self.month_med_ = month_price_med(data)
        self.global_med_ = get_median(data, "nightly_rate_idr")
        return self

    def fill_price_room_month(self, data, logs):
        count = 0
        for (room, month), med in self.room_month_med_.items():
            no_price = data["nightly_rate_idr"].isna()
            same_room = data["room_category"] == room
            same_month = data["arrival_month"] == month
            hit = no_price & same_room & same_month
            data.loc[hit, "nightly_rate_idr"] = med
            count += hit.sum()
        log_step(logs, 5, "NaN diisi median kamar di bulan x", "nightly_rate_idr", count)

    def fill_price_room(self, data, logs):
        count = 0
        for room, med in self.room_med_.items():
            no_price = data["nightly_rate_idr"].isna()
            same_room = data["room_category"] == room
            hit = no_price & same_room
            data.loc[hit, "nightly_rate_idr"] = med
            count += hit.sum()
        log_step(logs, 5, "NaN diisi median kamar", "nightly_rate_idr", count)

    def fill_price_month(self, data, logs):
        count = 0
        for month, med in self.month_med_.items():
            no_price = data["nightly_rate_idr"].isna()
            same_month = data["arrival_month"] == month
            hit = no_price & same_month
            data.loc[hit, "nightly_rate_idr"] = med
            count += hit.sum()
        log_step(logs, 5, "NaN diisi median semua kamar di bulan x", "nightly_rate_idr", count)

    def fill_price_global(self, data, logs):
        hit = data["nightly_rate_idr"].isna()
        data.loc[hit, "nightly_rate_idr"] = self.global_med_
        log_step(logs, 5, "NaN diisi median global", "nightly_rate_idr", hit.sum())

    def fill_price(self, data, logs):
        self.fill_price_room_month(data, logs)
        self.fill_price_room(data, logs)
        self.fill_price_month(data, logs)
        self.fill_price_global(data, logs)

    def transform(self, data, logs=None):
        if self.fill_values_ is None:
            raise RuntimeError("MissingValueImputer belum di-fit.")

        data = data.copy()

        self.fill_price(data, logs)

        for col in MODE_COLS:
            fill_col(data, col, self.fill_values_[col], "modus", logs)
        for col in MEDIAN_COLS:
            fill_col(data, col, self.fill_values_[col], "median", logs)
        return data

### Duplicates

This part covers repeated records.

split_data() sudah membuang duplikat fitur pada train

### Outliers

This part covers values far from the rest of the data.

Aturan domain mengubah nilai yang tidak valid menjadi nilai kosong sebelum imputasi. Aturannya ada di Data Errors.

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
import calendar

import numpy as np
import pandas as pd

FEATURE_COLS = [
    "num_adults",
    "num_children",
    "weekend_nights",
    "weekday_nights",
    "meal_package",
    "needs_parking",
    "room_category",
    "days_before_arrival",
    "arrival_year",
    "arrival_month",
    "arrival_day",
    "booking_channel",
    "is_returning_guest",
    "prior_cancellations",
    "prior_completed_bookings",
    "nightly_rate_idr",
    "special_requests",
]

CAT_COLS = [
    "meal_package",
    "room_category",
    "booking_channel",
]

NUM_COLS = [
    "num_adults",
    "num_children",
    "weekend_nights",
    "weekday_nights",
    "needs_parking",
    "days_before_arrival",
    "arrival_year",
    "arrival_month",
    "arrival_day",
    "is_returning_guest",
    "prior_cancellations",
    "prior_completed_bookings",
    "nightly_rate_idr",
    "special_requests",
]

INT_COLS = [
    "num_adults",
    "num_children",
    "weekend_nights",
    "weekday_nights",
    "needs_parking",
    "days_before_arrival",
    "arrival_year",
    "arrival_month",
    "arrival_day",
    "is_returning_guest",
    "prior_cancellations",
    "prior_completed_bookings",
    "special_requests",
]

NON_NEG_COLS = [
    "num_adults",
    "num_children",
    "weekend_nights",
    "weekday_nights",
    "days_before_arrival",
    "prior_cancellations",
    "prior_completed_bookings",
    "nightly_rate_idr",
    "special_requests",
]

BINARY_COLS = [
    "needs_parking",
    "is_returning_guest",
]

MIN_YEAR = 1
MIN_MONTH = 1
MAX_MONTH = 12
MIN_DAY = 1
SAFE_DAY = 28

def validate_data(data, valid_cats=None, logs=None):
    check_cols(data)
    data = data[FEATURE_COLS].copy()

    to_num_cols(data, logs)
    clean_cats(data, logs)
    if valid_cats is not None:
        new_cat_to_nan(data, valid_cats, logs)

    neg_to_nan(data, logs)
    decimal_to_nan(data, logs)
    bad_binary_to_nan(data, logs)
    zero_guests_to_nan(data, logs)
    bad_year_to_nan(data, logs)
    bad_month_to_nan(data, logs)
    bad_day_to_nan(data, logs)
    fix_dates(data, logs)
    return data

def log_step(logs, step, rule, col, n_rows):
    if logs is None:
        return
    logs.append(
        {
            "step": step,
            "rule": rule,
            "col": col,
            "n_rows": int(n_rows),
        }
    )


def set_nan(data, hit, col, step, rule, logs):
    data.loc[hit, col] = np.nan
    log_step(logs, step, rule, col, hit.sum())


def check_cols(data):
    missing = []
    double = []
    for col in FEATURE_COLS:
        if col not in data.columns:
            missing.append(col)
        elif list(data.columns).count(col) > 1:
            double.append(col)
    if missing:
        raise ValueError(f"Kolom fitur hilang: {missing}")
    if double:
        raise ValueError(f"Kolom fitur dobel: {double}")


def to_num(data, col, logs):
    nan_before = data[col].isna().sum()
    data[col] = pd.to_numeric(data[col], errors="coerce").astype("float64")
    nan_after = data[col].isna().sum()
    log_step(logs, 1, "teks di kolom angka jadi NaN", col, nan_after - nan_before)


def to_num_cols(data, logs):
    for col in NUM_COLS:
        to_num(data, col, logs)


def clean_cat(data, col, logs):
    new_values = []
    n_changed = 0
    for cell in data[col]:
        if pd.isna(cell):
            new_values.append(np.nan)
            continue
        text = str(cell).strip()
        if text != str(cell):
            n_changed += 1
        new_values.append(text)

    data[col] = pd.Series(new_values, index=data.index, dtype=object)
    log_step(logs, 1, "spasi di kategori dibuang", col, n_changed)

    empty_text = data[col] == ""
    set_nan(data, empty_text, col, 1, "teks kosong jadi NaN", logs)


def clean_cats(data, logs):
    for col in CAT_COLS:
        clean_cat(data, col, logs)


def new_cat_to_nan(data, valid_cats, logs):
    for col in CAT_COLS:
        has_value = data[col].notna()
        not_in_train = ~data[col].isin(valid_cats[col])
        hit = has_value & not_in_train
        set_nan(data, hit, col, 1, "kategori baru (gak ada di train) jadi NaN", logs)


def learn_cats(data):
    valid_cats = {}
    for col in CAT_COLS:
        cats = data[col].dropna().unique().tolist()
        if len(cats) == 0:
            raise ValueError(f"Kolom kategori '{col}' kosong semua di train split.")
        valid_cats[col] = cats
    return valid_cats


def neg_to_nan(data, logs):
    for col in NON_NEG_COLS:
        hit = data[col] < 0
        set_nan(data, hit, col, 2, "angka negatif jadi NaN", logs)


def not_int(data, col):
    return data[col].notna() & (data[col] % 1 != 0)


def decimal_to_nan(data, logs):
    for col in INT_COLS:
        hit = not_int(data, col)
        set_nan(data, hit, col, 2, "angka desimal di kolom bulat jadi NaN", logs)


def bad_binary_to_nan(data, logs):
    for col in BINARY_COLS:
        has_value = data[col].notna()
        not_binary = ~data[col].isin([0, 1])
        hit = has_value & not_binary
        set_nan(data, hit, col, 2, "nilai biner selain 0/1 jadi NaN", logs)


def zero_guests_to_nan(data, logs):
    no_adults = data["num_adults"] == 0
    no_kids = data["num_children"].isna() | (data["num_children"] == 0)
    hit = no_adults & no_kids
    set_nan(data, hit, "num_adults", 2, "tamu total 0 -> num_adults jadi NaN", logs)


def bad_year_to_nan(data, logs):
    hit = data["arrival_year"] < MIN_YEAR
    set_nan(data, hit, "arrival_year", 2, "tahun < 1 jadi NaN", logs)


def bad_month_to_nan(data, logs):
    month = data["arrival_month"]
    hit = month.notna() & ((month < MIN_MONTH) | (month > MAX_MONTH))
    set_nan(data, hit, "arrival_month", 2, "bulan di luar 1-12 jadi NaN", logs)


def bad_day_to_nan(data, logs):
    hit = data["arrival_day"] < MIN_DAY
    set_nan(data, hit, "arrival_day", 2, "hari < 1 jadi NaN", logs)


def clip_day(data):
    has_year = data["arrival_year"].notna()
    has_month = data["arrival_month"].notna()
    has_day = data["arrival_day"].notna()
    risky_day = data["arrival_day"] > SAFE_DAY
    cand = has_year & has_month & has_day & risky_day

    years = data.loc[cand, "arrival_year"]
    months = data.loc[cand, "arrival_month"]
    days = data.loc[cand, "arrival_day"]

    new_days = []
    count = 0
    for year, month, day in zip(years, months, days):
        last_day = calendar.monthrange(int(year), int(month))[1]
        if day > last_day:
            new_days.append(float(last_day))
            count += 1
        else:
            new_days.append(day)

    data.loc[cand, "arrival_day"] = new_days
    return count


def fix_dates(data, logs):
    count = clip_day(data)
    log_step(logs, 2, "tanggal gak ada di kalender -> akhir bulan", "arrival_day", count)


def recheck_dates(data, logs):
    count = clip_day(data)
    log_step(logs, 6, "cek ulang tanggal setelah diisi -> akhir bulan", "arrival_day", count)

In [ ]:
OUTPUT_COLS = FEATURE_COLS
REPORT_COLS = ["data", "step", "rule", "col", "n_rows"]


def fix_types(data):
    data = data[OUTPUT_COLS].copy()

    for col in INT_COLS:
        data[col] = data[col].astype("int64")
    for col in CAT_COLS:
        data[col] = data[col].astype(str)
    data["nightly_rate_idr"] = data["nightly_rate_idr"].astype("float64")
    return data


def final_check(data):
    n_nan = int(data.isna().sum().sum())
    if n_nan > 0:
        raise ValueError(f"Masih ada {n_nan} NaN setelah cleaning.")
    if list(data.columns) != OUTPUT_COLS:
        raise ValueError("Urutan kolom output gak sesuai kontrak.")


class DataCleaner:
    def __init__(self):
        self.valid_cats_ = None
        self.imputer_ = None
        self.logs_ = []

    def fit(self, data):
        clean = validate_data(data)
        valid_cats = learn_cats(clean)
        imputer = MissingValueImputer().fit(clean)

        self.valid_cats_ = valid_cats
        self.imputer_ = imputer
        self.logs_ = []
        return self

    def transform(self, data, name="data"):
        if self.valid_cats_ is None or self.imputer_ is None:
            raise RuntimeError("DataCleaner belum di-fit.")

        new_logs = []
        data = validate_data(data, self.valid_cats_, new_logs)
        data = self.imputer_.transform(data, new_logs)

        recheck_dates(data, new_logs)
        data = fix_types(data)
        final_check(data)

        for row in new_logs:
            row["data"] = name
            self.logs_.append(row)
        return data

    def fit_transform(self, data, name="data"):
        return self.fit(data).transform(data, name=name)

    def report(self):
        return pd.DataFrame(self.logs_, columns=REPORT_COLS)

    def save(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("wb") as f:
            pickle.dump(self, f)


def load_cleaner(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def cleaning(data, cleaner=None, fit=False, name="data"):
    if ID_COL in data.columns or TARGET_COL in data.columns:
        raise ValueError(f"data gak boleh punya kolom '{ID_COL}' atau '{TARGET_COL}'; pakai split_data() dulu.")

    if fit:
        if cleaner is None:
            cleaner = DataCleaner()
        data = cleaner.fit_transform(data, name=name)
    else:
        if cleaner is None:
            raise ValueError("Butuh `cleaner` yang udah di-fit kalau fit=False.")
        data = cleaner.transform(data, name=name)
    return data, cleaner

## Construct Data

Construction derives attributes and transforms values for modeling.

Pipeline bersama mengubah fitur awal menjadi representasi numerik.

### Attribute Construction

<!-- rubric v3: derived columns, rate_per_guest_night -->

This part covers new attributes derived from existing ones.

Pipeline bersama memakai fitur asli sebagai masukan. Fitur kategori diubah menjadi kolom one-hot, sedangkan fitur numerik distandardisasi.

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
def fit_scaler(X_train, num_cols):
    return StandardScaler().fit(X_train[num_cols])

def scale_features(X, scaler, num_cols):
    hasil = X.copy()
    hasil[num_cols] = scaler.transform(X[num_cols])
    return hasil

### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
def fit_encoder(X_train, cat_cols):
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoder.fit(X_train[cat_cols])
    return encoder


def encode_features(X, encoder, cat_cols):
    encoded = encoder.transform(X[cat_cols])
    encoded_features = pd.DataFrame(
        encoded,
        index=X.index,
        columns=encoder.get_feature_names_out(cat_cols),
    )
    numeric_features = X.drop(columns=cat_cols)
    return pd.concat([numeric_features, encoded_features], axis=1)

In [ ]:
# Write your code here

### Features Selection

In [ ]:
def mrmr(
    X: pd.DataFrame | np.ndarray,
    y: pd.Series | np.ndarray,
) -> dict[Hashable, float]:
    if np.ndim(X) != 2:
        raise ValueError("X harus berupa matriks fitur dua dimensi.")
    frame = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
    if frame.empty:
        raise ValueError("X harus memiliki setidaknya satu baris dan satu fitur.")
    if not frame.columns.is_unique:
        raise ValueError("Nama kolom pada X harus unik.")

    target = np.asarray(y)
    if target.ndim != 1 or len(target) != len(frame):
        raise ValueError("y harus satu dimensi dengan satu label untuk setiap baris X.")
    if isinstance(X, pd.DataFrame) and isinstance(y, pd.Series):
        if not X.index.equals(y.index):
            raise ValueError("Indeks baris X dan y harus sama dan memiliki urutan yang sama.")

    columns = [
        validateColumn(frame.iloc[:, index])
        for index in range(frame.shape[1])
    ]
    target = validateColumn(pd.Series(target, name="target"))
    relevance = np.array([
        mutualInformation(column, target) for column in columns
    ])

    redundancy = np.zeros((len(columns), len(columns)), dtype=float)
    for left in range(len(columns)):
        for right in range(left):
            value = mutualInformation(columns[left], columns[right])
            redundancy[left, right] = redundancy[right, left] = value

    ranking = iterateMID(relevance, redundancy)
    return {frame.columns[index]: score for index, score in ranking.items()}


def topK(scores: Mapping[Hashable, float], k: int) -> list[Hashable]:
    if isinstance(k, bool) or not isinstance(k, Integral):
        raise TypeError("k harus berupa bilangan bulat.")
    if not 1 <= k <= len(scores):
        raise ValueError("k harus berada di antara 1 dan jumlah fitur.")
    return list(scores)[:k]


def mid(relevance: float, redundancies: Sequence[float] = ()) -> float:
    penalty = float(np.mean(redundancies)) if len(redundancies) else 0.0
    return float(relevance) - penalty


def iterateMID(
    relevance: np.ndarray, redundancy: np.ndarray
) -> dict[int, float]:
    relevance = np.asarray(relevance, dtype=float)
    redundancy = np.asarray(redundancy, dtype=float)
    if relevance.ndim != 1 or relevance.size == 0:
        raise ValueError("relevance harus berupa array satu dimensi yang tidak kosong.")
    if redundancy.shape != (relevance.size, relevance.size):
        raise ValueError("redundancy harus berupa matriks persegi sesuai ukuran relevance.")
    if not np.isfinite(relevance).all() or not np.isfinite(redundancy).all():
        raise ValueError("Nilai informasi mutual harus terhingga.")
    if (relevance < 0).any() or (redundancy < 0).any():
        raise ValueError("Nilai informasi mutual tidak boleh negatif.")

    remaining = list(range(relevance.size))
    selected = []
    ranking = {}
    while remaining:
        scores = [
            mid(relevance[index], redundancy[index, selected])
            for index in remaining
        ]
        best = int(np.argmax(scores))
        index = remaining.pop(best)
        ranking[index] = scores[best]
        selected.append(index)
    return ranking


def validateColumn(column: pd.Series) -> np.ndarray:
    if column.isna().any():
        raise ValueError(
            f"Kolom {column.name!r} memiliki nilai yang hilang; bersihkan data terlebih dahulu."
        )
    if (not pd.api.types.is_numeric_dtype(column.dtype)
            or pd.api.types.is_complex_dtype(column.dtype)):
        raise ValueError(
            f"Kolom {column.name!r} harus berisi nilai numerik riil; "
            "siapkan data pada tahap transformasi terlebih dahulu."
        )
    if not np.isfinite(column.to_numpy(dtype=float)).all():
        raise ValueError(
            f"Kolom {column.name!r} memiliki nilai tidak terhingga; "
            "bersihkan data terlebih dahulu."
        )
    return column.to_numpy()


def mutualInformation(x: np.ndarray, y: np.ndarray) -> float:
    countX = Counter(x)
    countY = Counter(y)
    jointCounts = Counter(zip(x, y))
    sampleCount = len(x)
    score = sum(
        (count / sampleCount)
        * np.log((count * sampleCount) / (countX[left] * countY[right]))
        for (left, right), count in jointCounts.items()
    )
    return max(0.0, float(score))

def featureSelection(
    X: pd.DataFrame | np.ndarray,
    y: pd.Series | np.ndarray,
    k: int,
) -> tuple[pd.DataFrame | np.ndarray, list[Hashable]]:
    columns = topK(mrmr(X, y), k)
    if isinstance(X, pd.DataFrame):
        return X.loc[:, columns].copy(), columns
    return np.asarray(X)[:, columns].copy(), columns


### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [ ]:
def fitPCA(X, nComponents):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape[0] < 2 or X.shape[1] < 1:
        raise ValueError("PCA butuh matriks dua dimensi dengan minimal dua baris.")
    if not np.isfinite(X).all():
        raise ValueError("PCA butuh nilai numerik terhingga.")
    mean = X.mean(axis=0)
    _, S, Vt = np.linalg.svd(X - mean, full_matrices=False)
    variance = S**2
    total = variance.sum()
    ratio = variance / total if total > 0 else np.zeros_like(variance)
    if isinstance(nComponents, bool):
        raise TypeError("nComponents harus int atau float.")
    if isinstance(nComponents, Integral):
        if not 1 <= nComponents <= len(S):
            raise ValueError(f"nComponents harus di antara 1 dan {len(S)}.")
        k = int(nComponents)
    elif isinstance(nComponents, Real):
        if not 0 < nComponents < 1:
            raise ValueError("nComponents berupa float harus di antara 0 dan 1.")
        k = min(int(np.searchsorted(np.cumsum(ratio), nComponents) + 1), len(S))
    else:
        raise TypeError("nComponents harus int atau float.")
    return {"mean": mean, "components": Vt[:k], "explainedRatio": ratio[:k]}


def transformPCA(X, pca):
    return (np.asarray(X, dtype=float) - pca["mean"]) @ pca["components"].T

def dimensionality_reduction(X, reducer=None, fit=False, nComponents=None):
    if not isinstance(X, pd.DataFrame):
        raise ValueError("X harus berupa DataFrame.")
    if fit:
        if nComponents is None:
            raise ValueError("Butuh nComponents untuk fit PCA.")
        reducer = fitPCA(X, nComponents)
        reducer["inputColumns"] = X.columns.tolist()
    elif reducer is None:
        raise ValueError("Butuh reducer yang sudah di-fit.")
    if X.columns.tolist() != reducer["inputColumns"]:
        raise ValueError("Kolom data harus sama dengan kolom saat PCA dilatih.")
    values = transformPCA(X, reducer)
    columns = [f"PC{i + 1}" for i in range(values.shape[1])]
    return pd.DataFrame(values, index=X.index, columns=columns), reducer

## Class Balance

This part covers the proportion of each class in the training data.

In [ ]:
def handle_imbalance(X_train: pd.DataFrame, y_train: pd.Series, *, random_state: int = 42) -> tuple[pd.DataFrame, pd.Series]:
    if not X_train.index.equals(y_train.index):
        raise ValueError("Indeks fitur dan label train harus sama.")

    if not y_train.isin([0, 1]).all():
        raise ValueError("Label train harus berisi 0 atau 1.")

    counts = y_train.value_counts()
    if len(counts) != 2:
        raise ValueError("Oversampling membutuhkan kedua kelas di data train.")

    additional_rows = counts.max() - counts.min()
    if additional_rows == 0:
        return X_train.copy(), y_train.copy()

    rng = np.random.default_rng(random_state)
    minority_positions = np.flatnonzero(y_train.to_numpy() == counts.idxmin())
    sampled_positions = rng.choice(minority_positions, size=additional_rows, replace=True)
    positions = rng.permutation(np.concatenate((np.arange(len(y_train)), sampled_positions)))

    return X_train.iloc[positions].reset_index(drop=True), y_train.iloc[positions].reset_index(drop=True)

handleImbalance = handle_imbalance

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
def transformation(X, transformer=None, fit=False):
    if fit:
        transformer = {}
    elif transformer is None:
        raise ValueError("Butuh transformer yang sudah di-fit.")

    if not X.columns.is_unique:
        raise ValueError("Nama kolom fitur harus unik.")

    expected_cols = FEATURE_COLS if fit else transformer["input_cols"]
    missing_cols = [col for col in expected_cols if col not in X.columns]
    extra_cols = [col for col in X.columns if col not in expected_cols]
    if missing_cols or extra_cols:
        raise ValueError(f"Kolom fitur tidak sesuai. Kurang: {missing_cols}. Ekstra: {extra_cols}.")

    if X.isna().any().any():
        raise ValueError("Masih ada nilai kosong. Jalankan cleaning sebelum transformasi.")

    if not X[BINARY_COLS].isin([0, 1]).all().all():
        raise ValueError("Fitur biner harus berisi 0 atau 1 setelah cleaning.")

    if fit:
        num_cols = [col for col in NUM_COLS if col not in BINARY_COLS]
        transformer = {
            "encoder": fit_encoder(X, CAT_COLS),
            "scaler": fit_scaler(X, num_cols),
            "num_cols": num_cols,
            "input_cols": X.columns.tolist(),
        }

    encoded = encode_features(X, transformer["encoder"], CAT_COLS)
    scaled = scale_features(encoded, transformer["scaler"], transformer["num_cols"])

    if fit:
        transformer["columns"] = scaled.columns.tolist()
    return scaled.loc[:, transformer["columns"]], transformer


---

# Preparation Pipeline <a name="6"></a>

---

<!-- rubric v3: fitted steps, fitted_prep_steps -->

The pipeline combines the preparation steps so they run in one consistent order.

In [ ]:
class DTLPreprocessor:
    def fit(self, X):
        self.fit_transform(X)
        return self

    def fit_transform(self, X):
        self.cleaner = DataCleaner().fit(X)
        features = self._features(X)
        self.encoder = fit_encoder(features, CAT_COLS)
        encoded = encode_features(features, self.encoder, CAT_COLS).astype(float)
        self.columns = encoded.columns.tolist()
        return encoded

    def transform(self, X):
        features = self._features(X)
        return encode_features(features, self.encoder, CAT_COLS).loc[:, self.columns].astype(float)

    def _features(self, X):
        features = self.cleaner.transform(X)
        validated = validate_data(X)
        for column in validated.columns:
            features[column + "_missing"] = validated[column].isna().astype(float)
        for column in CAT_COLS:
            features[column] = validated[column].fillna("__missing__")
        arrival = pd.to_datetime({
            "year": features.arrival_year,
            "month": features.arrival_month,
            "day": features.arrival_day,
        })
        features["booking_ordinal"] = (
            (arrival - pd.Timestamp("2000-01-01")).dt.days - features.days_before_arrival
        )
        return features

In [ ]:
def runPreprocessing(
    X,
    y=None,
    *,
    fit=False,
    state=None,
    clean=True,
    transform=True,
    featureCount=None,
    continuousColumns=None,
    binCount=10,
    components=None,
    oversample=False,
    dtlPreprocessing=False,
    randomState=42,
    name="data",
):
    if not isinstance(X, pd.DataFrame) or X.empty or not X.columns.is_unique:
        raise ValueError("X harus berupa DataFrame yang tidak kosong dengan kolom unik.")
    if y is not None:
        if not isinstance(y, pd.Series) or not X.index.equals(y.index):
            raise ValueError("y harus berupa Series dengan indeks dan urutan yang sama dengan X.")
    if fit:
        if dtlPreprocessing and not (clean and transform):
            raise ValueError("DTL preprocessing membutuhkan cleaning dan transformation aktif.")
        state = {
            "version": 1,
            "inputColumns": X.columns.tolist(),
            "clean": clean,
            "transform": transform,
            "cleaner": None,
            "transformer": None,
            "outputColumns": None,
            "dtlPreprocessor": DTLPreprocessor() if dtlPreprocessing else None,
        }
    elif not isinstance(state, dict) or state.get("version") != 1:
        raise ValueError("Butuh state preprocessing yang sudah di-fit dan memiliki format yang sesuai.")
    if set(X.columns) != set(state["inputColumns"]):
        raise ValueError("Kolom data harus sama dengan kolom saat preprocessing dilatih.")
    features = X.loc[:, state["inputColumns"]].copy()
    target = None if y is None else y.copy()
    dtlPreprocessor = state.get("dtlPreprocessor")
    if dtlPreprocessor is not None:
        features = dtlPreprocessor.fit_transform(features) if fit else dtlPreprocessor.transform(features)
    elif state["clean"]:
        features, state["cleaner"] = cleaning(
            features, cleaner=state["cleaner"], fit=fit, name=name,
        )
    if dtlPreprocessor is None and state["transform"]:
        features, state["transformer"] = transformation(
            features, transformer=state["transformer"], fit=fit,
        )
    if fit:
        if featureCount is not None:
            if target is None:
                raise ValueError("Seleksi fitur membutuhkan label pelatihan.")
            selectionColumns = continuousColumns
            if selectionColumns is None:
                selectionColumns = [column for column in CONTINUOUS_COLS if column in features.columns]
                if dtlPreprocessor is not None:
                    selectionColumns.append("booking_ordinal")
            features, _ = featureSelection(
                features, target, featureCount,
                continuousColumns=selectionColumns, binCount=binCount,
            )
        state["selectedColumns"] = features.columns.tolist()
    else:
        features = features.loc[:, state.get("selectedColumns", state["outputColumns"])].copy()
    if fit:
        state["reducer"] = None
        if components is not None:
            features, state["reducer"] = dimensionality_reduction(
                features, fit=True, nComponents=components,
            )
        state["outputColumns"] = features.columns.tolist()
    elif state.get("reducer") is not None:
        features, _ = dimensionality_reduction(features, reducer=state["reducer"])
    if fit and oversample:
        if target is None:
            raise ValueError("Oversampling membutuhkan label pelatihan.")
        features, target = handleImbalance(features, target, random_state=randomState)
    return features, target, state

In [ ]:
X_train_ready, y_train_ready, prep = runPreprocessing(
    X_train, y_train, fit=True, name="train"
)

X_val_ready, y_val_ready, _ = runPreprocessing(
    X_val, y_val, state=prep, name="validation"
)

X_test_ready, _, _ = runPreprocessing(
    X_test, state=prep, name="test"
)

print("Train", X_train_ready.shape)
print("Validasi", X_val_ready.shape)
print("Test", X_test_ready.shape)

Train (21857, 30)
Validasi (5475, 30)
Test (6844, 30)


---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

<!-- rubric v3: pairwise distances, val_neighbor_dists -->

This part covers the nearest neighbor classifier and its validation.

In [ ]:
def compute_distance(a, b):
    """Compute the Euclidean distance between two feature rows."""
    if len(a) != len(b):
        raise ValueError("Jumlah fitur kedua baris harus sama.")

    total = 0
    for i in range(len(a)):
        selisih = a[i] - b[i]
        total += selisih * selisih
    return math.sqrt(total)


class KNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X_train, y_train):
        if len(X_train) != len(y_train):
            raise ValueError("Jumlah baris harus sama dengan jumlah label.")

        if not 1 <= self.k <= len(X_train):
            raise ValueError("k harus antara 1 dan jumlah baris latih.")

        if any(label not in (0, 1) for label in y_train):
            raise ValueError("Label KNN biner harus 0 atau 1.")

        self.X_train = X_train
        self.y_train = y_train
        return self

    def _predict_one(self, row):
        distances = []
        for i, train_row in enumerate(self.X_train):
            distance = compute_distance(row, train_row)
            distances.append((distance, i))

        distances.sort()
        k_hasil = distances[:self.k]
        labels = []
        for _, indeks in k_hasil:
            labels.append(self.y_train[indeks])
        suara_satu = labels.count(1)
        if suara_satu > len(labels) / 2:
            return 1
        return 0

    def predict(self, X):
        hasil = []
        for row in X:
            hasil.append(self._predict_one(row))
        return hasil

    def predict_proba(self, X):
        hasil = []

        for row in X:
            distances = []
            for i, train_row in enumerate(self.X_train):
                distance = compute_distance(row, train_row)
                distances.append((distance, i))
            distances.sort()
            k_hasil = distances[:self.k]
            labels = []
            for _, indeks in k_hasil:
                labels.append(self.y_train[indeks])
            p1 = labels.count(1) / len(labels)
            p0 = 1 - p1
            hasil.append([p0, p1])

        return hasil

In [ ]:
def nearest_labels(train, labels, queries, max_k, batch_size=16):
    train_columns = np.ascontiguousarray(train.T)
    result = np.empty((len(queries), max_k), dtype=labels.dtype)

    for start in range(0, len(queries), batch_size):
        batch = queries[start:start + batch_size]
        distances = np.zeros((len(batch), len(train)))
        delta = np.empty_like(distances)

        for query_col, train_col in zip(batch.T, train_columns):
            np.subtract(query_col[:, None], train_col, out=delta)
            np.square(delta, out=delta)
            np.add(distances, delta, out=distances)

        np.sqrt(distances, out=distances)

        # Untuk jarak yang sama, pertahankan urutan baris train.
        indices = np.argsort(distances, axis=1, kind="stable")[:, :max_k]
        result[start:start + len(batch)] = labels[indices]

    return result

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
def entropy(class_counts):
    counts = np.asarray(class_counts, dtype=float)
    totals = counts.sum(axis=-1, keepdims=True)
    probabilities = np.divide(counts, totals, out=np.zeros_like(counts), where=totals > 0)
    logarithms = np.log2(np.where(probabilities > 0, probabilities, 1))
    return -np.sum(probabilities * logarithms, axis=-1)


def evaluate_split(parent_counts, left_counts, right_counts):
    parent = np.asarray(parent_counts, dtype=float)
    left = np.asarray(left_counts, dtype=float)
    right = np.asarray(right_counts, dtype=float)
    weighted = left.sum(axis=-1) * entropy(left) + right.sum(axis=-1) * entropy(right)
    remainder = np.divide(
        weighted, parent.sum(axis=-1), out=np.zeros_like(weighted), where=parent.sum(axis=-1) > 0,
    )
    return entropy(parent) - remainder


@dataclass
class TreeNode:
    class_counts: np.ndarray
    feature_index: int | None = None
    split_value: float | None = None
    left: "TreeNode | None" = None
    right: "TreeNode | None" = None


class DecisionTreeClassifier:
    def __init__(
        self, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, min_impurity_decrease=0.0,
    ):
        for name, value, minimum in (
            ("max_depth", max_depth, 0),
            ("min_samples_split", min_samples_split, 2),
            ("min_samples_leaf", min_samples_leaf, 1),
        ):
            if name == "max_depth" and value is None:
                continue
            if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
                raise ValueError(f"{name} harus berupa integer >= {minimum}.")
        if not np.isfinite(min_impurity_decrease) or min_impurity_decrease < 0:
            raise ValueError("min_impurity_decrease harus finite dan nonnegatif.")
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_impurity_decrease = min_impurity_decrease
        self.tree_ = None

    @staticmethod
    def _features(X):
        values = np.asarray(X, dtype=float)
        if values.ndim != 2 or values.shape[1] == 0 or not np.isfinite(values).all():
            raise ValueError("X harus berupa matriks numerik tanpa null; jalankan preprocessing.")
        return values

    def fit(self, X, y):
        self.tree_ = None
        values, target = self._features(X), np.asarray(y)
        if target.ndim != 1 or len(target) != len(values) or len(target) == 0:
            raise ValueError("y harus satu dimensi dan sesuai jumlah baris X.")
        if not np.isin(target, [0, 1]).all():
            raise ValueError("Label is_canceled harus 0 atau 1.")
        self.classes_ = np.array([0, 1])
        self.n_features_in_ = values.shape[1]
        self.n_samples_ = len(values)
        self.tree_ = self._build_tree(values, target.astype(int), 0)
        return self

    def _best_split(self, X, y, counts):
        best = (None, None, -np.inf)
        for feature in range(self.n_features_in_):
            order = np.argsort(X[:, feature], kind="stable")
            values = X[order, feature]
            positions = np.flatnonzero(values[:-1] < values[1:])
            sizes = positions + 1
            valid = (sizes >= self.min_samples_leaf) & (len(y) - sizes >= self.min_samples_leaf)
            positions, sizes = positions[valid], sizes[valid]
            if len(positions) == 0:
                continue
            positives = np.cumsum(y[order])[positions]
            left = np.column_stack((sizes - positives, positives))
            gains = evaluate_split(counts, left, counts - left)
            index = int(np.argmax(gains))
            if gains[index] > best[2]:
                position = positions[index]
                low, high = values[position], values[position + 1]
                threshold = low / 2 + high / 2
                if not low <= threshold < high:
                    threshold = low
                best = (feature, float(threshold), float(gains[index]))
        return best

    def _build_tree(self, X, y, depth):
        counts = np.bincount(y, minlength=2)
        node = TreeNode(counts)
        if (
            np.count_nonzero(counts) == 1
            or (self.max_depth is not None and depth >= self.max_depth)
            or len(y) < max(self.min_samples_split, 2 * self.min_samples_leaf)
        ):
            return node
        feature, threshold, gain = self._best_split(X, y, counts)
        if feature is None or len(y) / self.n_samples_ * gain + 1e-12 < self.min_impurity_decrease:
            return node
        left = X[:, feature] <= threshold
        node.feature_index, node.split_value = feature, threshold
        node.left = self._build_tree(X[left], y[left], depth + 1)
        node.right = self._build_tree(X[~left], y[~left], depth + 1)
        return node

    def predict_proba(self, X):
        if self.tree_ is None:
            raise RuntimeError("Latih model dengan fit() sebelum prediksi.")
        values = self._features(X)
        if values.shape[1] != self.n_features_in_:
            raise ValueError("Jumlah fitur prediksi harus sama dengan saat training.")
        probabilities = np.empty((len(values), 2))
        for index, row in enumerate(values):
            node = self.tree_
            while node.feature_index is not None:
                node = node.left if row[node.feature_index] <= node.split_value else node.right
            probabilities[index] = node.class_counts / node.class_counts.sum()
        return probabilities

    def predict(self, X):
        probabilities = self.predict_proba(X)
        return self.classes_[np.argmax(probabilities, axis=1)]


DTL = DecisionTreeClassifier


def dtl(X, y, **parameters):
    return DTL(**parameters).fit(X, y)

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [ ]:
@dataclass
class LogisticRegression:
    weights: np.ndarray
    bias: float
    lossHistory: list[float]
    parameterHistory: np.ndarray | None = None
    trainingFeatures: np.ndarray | None = None
    trainingTarget: np.ndarray | None = None

    def predictProba(self, X: np.ndarray) -> np.ndarray:
        values = validateFeatures(X)
        if values.shape[1] != len(self.weights):
            raise ValueError("Jumlah fitur prediksi harus sama dengan jumlah fitur pelatihan.")
        try:
            with np.errstate(over="raise", invalid="raise"):
                scores = values @ self.weights + self.bias
                if not np.isfinite(scores).all():
                    raise FloatingPointError
                positive = sigmoid(scores)
        except FloatingPointError as error:
            raise ValueError("Perhitungan prediksi menghasilkan nilai tidak terhingga.") from error
        return np.column_stack((1.0 - positive, positive))

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        threshold = validateScalar(threshold, "threshold")
        if not 0.0 <= threshold <= 1.0:
            raise ValueError("threshold harus berada di antara 0 dan 1.")
        return (self.predictProba(X)[:, 1] >= threshold).astype(int)


def sigmoid(scores: np.ndarray | float) -> np.ndarray:
    with np.errstate(under="ignore"):
        return np.exp(-np.logaddexp(0.0, -np.asarray(scores, dtype=float)))


def sga(
    weights: np.ndarray,
    bias: float,
    row: np.ndarray,
    target: float,
    learningRate: float = 0.01,
) -> tuple[np.ndarray, float]:
    error = target - float(sigmoid(weights @ row + bias))
    return weights + learningRate * error * row, float(bias + learningRate * error)


def sgaL1(
    weights: np.ndarray,
    bias: float,
    row: np.ndarray,
    target: float,
    learningRate: float = 0.01,
    regularizationStrength: float = 0.01,
) -> tuple[np.ndarray, float]:
    updated, updatedBias = sga(weights, bias, row, target, learningRate)
    updated -= learningRate * regularizationStrength * np.sign(weights)
    return updated, updatedBias


def sgaL2(
    weights: np.ndarray,
    bias: float,
    row: np.ndarray,
    target: float,
    learningRate: float = 0.01,
    regularizationStrength: float = 0.01,
) -> tuple[np.ndarray, float]:
    updated, updatedBias = sga(weights, bias, row, target, learningRate)
    updated -= learningRate * regularizationStrength * weights
    return updated, updatedBias


def sgaElasticNet(
    weights: np.ndarray,
    bias: float,
    row: np.ndarray,
    target: float,
    learningRate: float = 0.01,
    regularizationStrength: float = 0.01,
    l1Ratio: float = 0.5,
) -> tuple[np.ndarray, float]:
    updated, updatedBias = sga(weights, bias, row, target, learningRate)
    penalty = l1Ratio * np.sign(weights) + (1.0 - l1Ratio) * weights
    updated -= learningRate * regularizationStrength * penalty
    return updated, updatedBias


def iterateSGA(
    X: np.ndarray,
    y: np.ndarray,
    *,
    learningRate: float = 0.01,
    epochs: int = 100,
    regularization: str | None = "none",
    regularizationStrength: float = 0.01,
    l1Ratio: float = 0.5,
    shuffle: bool = True,
    randomState: int | None = 42,
    recordHistory: bool = False,
) -> LogisticRegression:
    values = validateFeatures(X)
    target = validateTarget(y, len(values))
    learningRate = validateScalar(learningRate, "learningRate")
    regularizationStrength = validateScalar(regularizationStrength, "regularizationStrength")
    l1Ratio = validateScalar(l1Ratio, "l1Ratio")
    if learningRate <= 0:
        raise ValueError("learningRate harus lebih besar dari 0.")
    if regularizationStrength < 0:
        raise ValueError("regularizationStrength tidak boleh negatif.")
    if not 0.0 <= l1Ratio <= 1.0:
        raise ValueError("l1Ratio harus berada di antara 0 dan 1.")
    if isinstance(epochs, bool) or not isinstance(epochs, Integral) or epochs < 1:
        raise ValueError("epochs harus berupa bilangan bulat positif.")
    if not isinstance(shuffle, (bool, np.bool_)):
        raise TypeError("shuffle harus berupa nilai boolean.")
    if not isinstance(recordHistory, (bool, np.bool_)):
        raise TypeError("recordHistory harus berupa nilai boolean.")
    if randomState is not None:
        if isinstance(randomState, bool) or not isinstance(randomState, Integral) or randomState < 0:
            raise ValueError("randomState harus berupa bilangan bulat nonnegatif atau None.")
        randomState = int(randomState)
    if regularization is None:
        regularization = "none"
    if not isinstance(regularization, str):
        raise TypeError("regularization harus berupa teks atau None.")
    regularization = regularization.lower()
    if regularization not in ("none", "l1", "l2", "elasticnet"):
        raise ValueError("regularization harus berupa 'none', 'l1', 'l2', atau 'elasticnet'.")

    update = sga
    l1Strength = 0.0
    l2Strength = 0.0
    if regularization == "l1":
        update = partial(sgaL1, regularizationStrength=regularizationStrength)
        l1Strength = regularizationStrength
    elif regularization == "l2":
        update = partial(sgaL2, regularizationStrength=regularizationStrength)
        l2Strength = regularizationStrength
    elif regularization == "elasticnet":
        update = partial(
            sgaElasticNet,
            regularizationStrength=regularizationStrength,
            l1Ratio=l1Ratio,
        )
        l1Strength = regularizationStrength * l1Ratio
        l2Strength = regularizationStrength * (1.0 - l1Ratio)

    weights = np.zeros(values.shape[1], dtype=float)
    bias = 0.0
    lossHistory = [logLoss(values @ weights + bias, target, weights, l1Strength, l2Strength)]
    parameterHistory = [np.concatenate(([bias], weights))] if recordHistory else None
    generator = np.random.default_rng(randomState)
    try:
        with np.errstate(over="raise", invalid="raise", divide="raise"):
            for epoch in range(epochs):
                order = generator.permutation(len(values)) if shuffle else range(len(values))
                for index in order:
                    weights, bias = update(
                        weights, bias, values[index], target[index], learningRate
                    )
                loss = logLoss(values @ weights + bias, target, weights, l1Strength, l2Strength)
                if not np.isfinite(loss) or not np.isfinite(weights).all() or not np.isfinite(bias):
                    raise FloatingPointError
                lossHistory.append(loss)
                if recordHistory:
                    parameterHistory.append(np.concatenate(([bias], weights)))
    except FloatingPointError as error:
        raise ValueError(
            "Pelatihan menghasilkan nilai tidak terhingga; "
            "gunakan data yang sudah diskalakan atau learningRate yang lebih kecil."
        ) from error
    return LogisticRegression(
        weights=weights, bias=bias, lossHistory=lossHistory,
        parameterHistory=np.asarray(parameterHistory) if recordHistory else None,
        trainingFeatures=values.copy() if recordHistory else None,
        trainingTarget=target.copy() if recordHistory else None,
    )


def logLoss(
    scores: np.ndarray,
    target: np.ndarray,
    weights: np.ndarray,
    l1Strength: float = 0.0,
    l2Strength: float = 0.0,
) -> float:
    likelihoodLoss = np.mean(np.logaddexp(0.0, (1.0 - 2.0 * target) * scores))
    penalty = l1Strength * np.sum(np.abs(weights)) + 0.5 * l2Strength * (weights @ weights)
    return float(likelihoodLoss + penalty)


def validateFeatures(X: np.ndarray) -> np.ndarray:
    try:
        values = np.asarray(X)
        if np.iscomplexobj(values) or values.dtype.kind in "USV":
            raise ValueError
        values = values.astype(float)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError("X harus berisi nilai numerik riil yang sudah ditransformasi.") from error
    if values.ndim != 2 or values.shape[0] == 0 or values.shape[1] == 0:
        raise ValueError("X harus berupa matriks dua dimensi dengan baris dan fitur yang tidak kosong.")
    if not np.isfinite(values).all():
        raise ValueError("X tidak boleh memiliki nilai yang hilang atau tidak terhingga.")
    return values


def validateTarget(y: np.ndarray, sampleCount: int) -> np.ndarray:
    try:
        target = np.asarray(y)
        if np.iscomplexobj(target) or target.dtype.kind in "USV":
            raise ValueError
        target = target.astype(float)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError("y harus berisi label numerik 0 dan 1.") from error
    if target.ndim != 1 or len(target) != sampleCount:
        raise ValueError("y harus satu dimensi dengan satu label untuk setiap baris X.")
    if not np.isfinite(target).all() or not np.isin(target, [0.0, 1.0]).all():
        raise ValueError("y hanya boleh berisi label 0 dan 1 tanpa nilai yang hilang.")
    if np.unique(target).size != 2:
        raise ValueError("Data pelatihan harus memiliki kedua kelas, yaitu 0 dan 1.")
    return target


def validateScalar(value: float, name: str) -> float:
    if isinstance(value, bool) or not isinstance(value, Real) or not np.isfinite(value):
        raise ValueError(f"{name} harus berupa bilangan riil terhingga.")
    return float(value)


def logisticRegression(
    X: np.ndarray,
    y: np.ndarray,
    regularization: str | None = "none",
    *,
    learningRate: float = 0.01,
    epochs: int = 100,
    regularizationStrength: float = 0.01,
    l1Ratio: float = 0.5,
    shuffle: bool = True,
    randomState: int | None = 42,
    recordHistory: bool = False,
) -> LogisticRegression:
    return iterateSGA(
        X,
        y,
        learningRate=learningRate,
        epochs=epochs,
        regularization=regularization,
        regularizationStrength=regularizationStrength,
        l1Ratio=l1Ratio,
        shuffle=shuffle,
        randomState=randomState,
        recordHistory=recordHistory,
    )

## Support Vector Machine

<!-- rubric v3: total slack, slack_total -->

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
  # --- Kernels ---
  def linear_kernel(A, B):
      return A @ B.T

  def polynomial_kernel(A, B, gamma, coef0, degree):
      return (gamma * (A @ B.T) + coef0) ** degree

  def gaussian_kernel(A, B, gamma):
      sq = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * (A @ B.T)
      return np.exp(-gamma * np.maximum(sq, 0.0))

  def compute_kernel(A, B, kernel="linear", gamma=1.0, coef0=0.0, degree=3):
      if kernel == "linear":
          return linear_kernel(A, B)
      if kernel == "poly":
          return polynomial_kernel(A, B, gamma, coef0, degree)
      if kernel == "rbf":
          return gaussian_kernel(A, B, gamma)
      raise ValueError(f"unknown kernel {kernel!r}; use 'linear', 'poly', 'rbf'")


  class SVM:
      def __init__(self, C=1.0, kernel="rbf", degree=3, gamma="scale", coef0=0.0,
                   tol=1e-3, max_passes=5, max_iter=100_000, random_state=0, class_weight=None):
          self.C, self.kernel, self.degree = C, kernel, degree
          self.gamma, self.coef0, self.tol = gamma, coef0, tol
          self.max_passes, self.max_iter, self.random_state = max_passes, max_iter, random_state
          self.class_weight = class_weight

      def _K(self, A, B):
          return compute_kernel(A, B, self.kernel, self.gamma_, self.coef0, self.degree)

      def fit(self, X, y):
          X = np.asarray(X, dtype=float)
          y = np.asarray(y)
          self.classes_ = np.unique(y)
          if len(self.classes_) != 2:
              raise ValueError("SVM is binary; got %d classes" % len(self.classes_))
          t = np.where(y == self.classes_[1], 1.0, -1.0)
          self.gamma_ = (1.0 / (X.shape[1] * X.var() or 1.0)) if self.gamma == "scale" else float(self.gamma)

          n, tol = len(X), self.tol
          cw = self.class_weight
          if cw == "balanced":
              cw = {k: n / (2 * np.sum(y == k)) for k in self.classes_}
          C = self.C * (np.array([cw[k] for k in y], dtype=float) if cw else np.ones(n))
          rng = np.random.default_rng(self.random_state)
          a, b = np.zeros(n), 0.0
          E = -t
          col = lambda i: self._K(X, X[i:i + 1])[:, 0]
          diag = np.array([self._K(x[None], x[None])[0, 0] for x in X])

          def step(i, j, Ki):
              nonlocal b
              if i == j:
                  return False
              if t[i] != t[j]:
                  L, H = max(0.0, a[j] - a[i]), min(C[j], C[i] + a[j] - a[i])
              else:
                  L, H = max(0.0, a[i] + a[j] - C[i]), min(C[j], a[i] + a[j])
              eta = 2 * Ki[j] - Ki[i] - diag[j]
              if L == H or eta >= 0:
                  return False
              aj = np.clip(a[j] - t[j] * (E[i] - E[j]) / eta, L, H)
              if abs(aj - a[j]) < 1e-8:
                  return False
              Kj = col(j)
              ai = a[i] + t[i] * t[j] * (a[j] - aj)
              di, dj = ai - a[i], aj - a[j]
              b1 = b - E[i] - t[i] * di * Ki[i] - t[j] * dj * Ki[j]
              b2 = b - E[j] - t[i] * di * Ki[j] - t[j] * dj * Kj[j]
              nb = b1 if 0 < ai < C[i] else b2 if 0 < aj < C[j] else (b1 + b2) / 2
              E[:] += t[i] * di * Ki + t[j] * dj * Kj + (nb - b)
              a[i], a[j], b = ai, aj, nb
              return True

          passes = it = 0
          while passes < self.max_passes and it < self.max_iter:
              changed = 0
              for i in range(n):
                  if (t[i] * E[i] < -tol and a[i] < C[i]) or (t[i] * E[i] > tol and a[i] > 0):
                      Ki = col(i)
                      js = [int(np.argmax(np.abs(E[i] - E)))]
                      js += list(np.roll(np.arange(n), -int(rng.integers(n))))
                      if any(step(i, j, Ki) for j in js):
                          changed += 1
                  it += 1
                  if it >= self.max_iter:
                      break
              passes = passes + 1 if changed == 0 else 0

          sv = a > 1e-8
          self.support_vectors_, self.dual_coef_ = X[sv], (a * t)[sv]
          self.support_, self.alpha_, self.b_ = np.flatnonzero(sv), a[sv], b
          return self

      def decision_function(self, X):
          return self._K(np.asarray(X, dtype=float), self.support_vectors_) @ self.dual_coef_ + self.b_

      def predict(self, X):
          return self.classes_[(self.decision_function(X) >= 0).astype(int)]

      def save(self, path):
          with open(path, "wb") as f:
              pickle.dump(self, f)

      @staticmethod
      def load(path):
          with open(path, "rb") as f:
              return pickle.load(f)

  # Hyperparameter final
  SVM_PARAMS = dict(C=10, kernel="rbf", gamma=0.2, max_iter=800_000, random_state=42)

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
# Sumber: https://scikit-learn.org/stable/modules/naive_bayes.html#gaussian-naive-bayes
#         https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html

class NaiveBayes:
    def __init__(self, smoothing=1e-9):
        self.smoothing = smoothing
        self.classes = None
        self.prior_logs = None
        self.means = None
        self.variances = None
        self.used_cols = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        if X.ndim != 2:
            raise ValueError("X harus berupa matriks dua dimensi.")
        if len(X) != len(y):
            raise ValueError("Jumlah baris X dan y harus sama.")

        self.classes = np.unique(y)
        feature_count = X.shape[1] #kolom
        class_count = len(self.classes)

        self.prior_logs = np.zeros(class_count)
        self.means = np.zeros((class_count, feature_count))
        self.variances = np.zeros((class_count, feature_count))

        for i in range (class_count):
            rows = X[y == self.classes[i]]
            self.prior_logs[i] = np.log(len(rows) / len(X))
            self.means[i] = rows.mean(axis=0)
            self.variances[i] = rows.var(axis=0)

        largest_variance = X.var(axis=0).max()
        self.variances = self.variances + self.smoothing * largest_variance

        self.used_cols = np.arange(feature_count)
        return self

    def class_scores(self, X):
        if self.classes is None:
            raise ValueError("Model belum dilatih, panggil fit terlebih dahulu.")

        X = np.asarray(X, dtype=float)
        X = X[:, self.used_cols]
        means = self.means[:, self.used_cols]
        variances = self.variances[:, self.used_cols]

        # Turunan rumus log Gaussian (var = sigma^2, mu = mean kelas):
        #   p(x)     = 1 / sqrt(2*pi*var) * exp(-(x - mu)^2 / (2*var))
        #   log p(x) = log(1 / sqrt(2*pi*var)) + log(exp(-(x - mu)^2 / (2*var)))
        #            = log((2*pi*var)^(-1/2)) - (x - mu)^2 / (2*var)
        #            = -0.5 * log(2*pi*var) - (x - mu)^2 / (2*var)

        scores = np.zeros((len(X), len(self.classes)))
        for i in range (len(self.classes)):
            log_likelihood = -0.5 * np.log(2 * np.pi * variances[i]) - ((X - means[i]) ** 2) / (2 * variances[i])
            scores[:, i] = self.prior_logs[i] + log_likelihood.sum(axis=1)
        return scores

    def predict_proba(self, X):
        scores = self.class_scores(X)
        largest = scores.max(axis=1, keepdims=True)
        values = np.exp(scores - largest) # Mencegah largest score gabisa dihitung pake exp (pasti exp(0))
        total = values.sum(axis=1, keepdims=True)
        return values / total

    def predict(self, X):
        scores = self.class_scores(X)
        best = np.argmax(scores, axis=1)
        return self.classes[best]


def naive_bayes(X, y, smoothing=1e-9, pick_features=False):
    model = NaiveBayes(smoothing=smoothing)
    model.fit(X, y)

    if pick_features:
        picked, round_scores = select_features(X, y, smoothing=smoothing)
        model.used_cols = picked
        model.round_scores = round_scores

    return model


def split_bayes_data(X, y, seed):
    rng = np.random.default_rng(seed)
    train_rows = []
    check_rows = []

    for label in np.unique(y):
        rows = np.arange(len(y))[y == label]
        rng.shuffle(rows)
        check_count = round(len(rows) * 0.25)
        if check_count == 0:
            check_count = 1
        check_rows.extend(rows[:check_count])
        train_rows.extend(rows[check_count:])

    train_rows = np.array(train_rows)
    check_rows = np.array(check_rows)
    return X[train_rows], y[train_rows], X[check_rows], y[check_rows]


def macro_f1(y_true, y_pred):
    cancel_f1 = evaluateMetrics(y_pred, y_true)["f1Score"]
    stay_f1 = evaluateMetrics(1 - y_pred, 1 - y_true)["f1Score"]
    return (cancel_f1 + stay_f1) / 2


def pick_features(X, y, smoothing=1e-9, seed=42):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    if X.ndim != 2 or X.shape[1] == 0:
        raise ValueError("X harus berupa matriks dua dimensi dengan minimal satu kolom.")
    if len(X) != len(y):
        raise ValueError("Jumlah baris X dan y harus sama.")

    train_X, train_y, check_X, check_y = split_bayes_data(X, y, seed)

    model = NaiveBayes(smoothing=smoothing)
    model.fit(train_X, train_y)

    picked = []
    round_scores = []
    previous_score = -1.0

    while len(picked) < X.shape[1]:
        best_col = None
        best_score = -1.0

        for col in range(X.shape[1]):
            if col in picked:
                continue
            model.used_cols = picked + [col]
            score = macro_f1(check_y, model.predict(check_X))
            if score > best_score:
                best_score = score
                best_col = col

        if best_score <= previous_score:
            break

        picked.append(best_col)
        round_scores.append(best_score)
        previous_score = best_score

    return picked, round_scores

select_features = pick_features

## Metrik dan pelatihan validasi

Kode metrik dari integrate dipakai untuk memperoleh F1 binary kelas batal dan Macro F1. Tabel hasil memuat jumlah baris training agar cakupan SVM terlihat.

In [ ]:
def confusionCounts(result, testData) -> tuple[int, int, int, int]:
    try:
        predicted = np.asarray(result)
        actual = np.asarray(testData)
    except (TypeError, ValueError) as error:
        raise ValueError("Prediksi dan label asli harus berupa array satu dimensi.") from error
    if predicted.ndim != 1 or actual.ndim != 1 or not len(actual):
        raise ValueError("Prediksi dan label asli harus berupa array satu dimensi yang tidak kosong.")
    if len(predicted) != len(actual):
        raise ValueError("Jumlah prediksi harus sama dengan jumlah label asli.")
    for values in (predicted, actual):
        if values.dtype.kind not in "biuf" or not np.isfinite(values).all():
            raise ValueError("Prediksi dan label asli harus berupa label numerik 0 dan 1 yang terhingga.")
        if not np.isin(values, [0, 1]).all():
            raise ValueError("Prediksi dan label asli hanya boleh berisi label 0 dan 1.")
    trueNegative = int(np.sum((actual == 0) & (predicted == 0)))
    falsePositive = int(np.sum((actual == 0) & (predicted == 1)))
    falseNegative = int(np.sum((actual == 1) & (predicted == 0)))
    truePositive = int(np.sum((actual == 1) & (predicted == 1)))
    return trueNegative, falsePositive, falseNegative, truePositive


def evaluateMetrics(result, testData) -> dict[str, float]:
    trueNegative, falsePositive, falseNegative, truePositive = confusionCounts(result, testData)
    total = trueNegative + falsePositive + falseNegative + truePositive
    predictedPositive = truePositive + falsePositive
    actualPositive = truePositive + falseNegative
    actualNegative = trueNegative + falsePositive
    f1Denominator = 2 * truePositive + falsePositive + falseNegative
    negativeF1Denominator = 2 * trueNegative + falsePositive + falseNegative
    positiveF1 = 2 * truePositive / f1Denominator if f1Denominator else 0.0
    negativeF1 = 2 * trueNegative / negativeF1Denominator if negativeF1Denominator else 0.0
    positiveRecall = truePositive / actualPositive if actualPositive else 0.0
    classRecalls = []
    if actualPositive:
        classRecalls.append(positiveRecall)
    if actualNegative:
        classRecalls.append(trueNegative / actualNegative)
    return {
        "accuracy": (truePositive + trueNegative) / total,
        "precision": truePositive / predictedPositive if predictedPositive else 0.0,
        "recall": positiveRecall,
        "f1Score": positiveF1,
        "macrof1Score": (positiveF1 + negativeF1) / 2,
        "balancedAccuracy": sum(classRecalls) / len(classRecalls),
    }


def accuracy(result, testData) -> float:
    return evaluateMetrics(result, testData)["accuracy"]


def precision(result, testData) -> float:
    return evaluateMetrics(result, testData)["precision"]


def recall(result, testData) -> float:
    return evaluateMetrics(result, testData)["recall"]


def f1Score(result, testData) -> float:
    return evaluateMetrics(result, testData)["f1Score"]


def balancedAccuracy(result, testData) -> float:
    return evaluateMetrics(result, testData)["balancedAccuracy"]


def macrof1Score(result, testData) -> float:
    return evaluateMetrics(result, testData)["macrof1Score"]

In [ ]:
def svm(X, y, **parameters):
    return SVM(**parameters).fit(X, y)

In [ ]:
train_array = X_train_ready.to_numpy(dtype=float)
train_labels = y_train_ready.to_numpy(dtype=int)
val_array = X_val_ready.to_numpy(dtype=float)
val_labels = y_val_ready.to_numpy(dtype=int)

TRAINERS = {
    "knn": lambda X, y, **params: KNN(**params).fit(X.tolist(), y.tolist()),
    "dtl": dtl,
    "logistic_regression": logisticRegression,
    "svm": svm,
    "naive_bayes": naive_bayes,
}
models = {}
predictions = {}
model_parameters = {}
training_inputs = {}


def notebook_training_rows(name, features, labels):
    if name != "svm" or SVM_TRAIN_LIMIT is None or len(features) <= SVM_TRAIN_LIMIT:
        return features, labels
    selected, _ = train_test_split(
        np.arange(len(labels)),
        train_size=SVM_TRAIN_LIMIT,
        stratify=labels,
        random_state=SEED,
    )
    selected.sort()
    return features[selected], labels[selected]


def notebook_predict(name, model, features):
    if name == "knn":
        neighbours = nearest_labels(
            np.asarray(model.X_train, dtype=float),
            np.asarray(model.y_train, dtype=int),
            features,
            model.k,
        )
        return (neighbours.sum(axis=1) > model.k / 2).astype(int)
    if name == "svm":
        return np.concatenate([
            model.predict(features[start:start + 256])
            for start in range(0, len(features), 256)
        ]).astype(int)
    return np.asarray(model.predict(features), dtype=int)


def fit_validation_model(name):
    options = dict(NOTEBOOK_CONFIG[name])
    features, labels = notebook_training_rows(name, train_array, train_labels)
    print(f"Melatih {name} dengan {len(labels):,} baris training.", flush=True)
    model = TRAINERS[name](features, labels, **options)
    result = notebook_predict(name, model, val_array)
    models[name] = model
    predictions[name] = result
    model_parameters[name] = options
    training_inputs[name] = (features, labels)
    print(f"Selesai {name}.", flush=True)

### Validasi KNN

In [ ]:
k_values = tuple(k for k in NOTEBOOK_CONFIG["knn_candidates"] if 1 <= k <= len(train_labels))
if not k_values:
    raise ValueError("Tidak ada nilai k yang valid untuk jumlah baris train.")

print(f"Mencari tetangga KNN untuk {len(val_labels):,} baris validasi.", flush=True)
knn_neighbors = nearest_labels(train_array, train_labels, val_array, max(k_values))
knn_candidates = {
    k: (knn_neighbors[:, :k].sum(axis=1) > k / 2).astype(int)
    for k in k_values
}
knn_tuning = pd.DataFrame([
    {"k": k, **evaluateMetrics(result, val_labels)}
    for k, result in knn_candidates.items()
])
best_k = int(knn_tuning.loc[knn_tuning["f1Score"].idxmax(), "k"])
knn = TRAINERS["knn"](train_array, train_labels, k=best_k)
np.testing.assert_array_equal(knn.predict(val_array[:4].tolist()), knn_candidates[best_k][:4])
models["knn"] = knn
predictions["knn"] = knn_candidates[best_k]
model_parameters["knn"] = {"k": best_k}
training_inputs["knn"] = (train_array, train_labels)
print(f"k terbaik menurut F1 binary pada validasi: {best_k}.")
display(knn_tuning.sort_values("f1Score", ascending=False).head(10).round(4))

Mencari tetangga KNN untuk 5,475 baris validasi.
k terbaik menurut F1 binary pada validasi: 5.


,k,accuracy,precision,recall,f1Score,macrof1Score,balancedAccuracy
3,5,0.8353,0.7611,0.7246,0.7424,0.8107,0.8069
11,13,0.8358,0.7691,0.7129,0.7399,0.8100,0.8043
13,15,0.8347,0.7660,0.7135,0.7388,0.8090,0.8036
5,7,0.8345,0.7684,0.7085,0.7372,0.8082,0.8022
9,11,0.8336,0.7642,0.7118,0.7371,0.8077,0.8024
1,3,0.8274,0.7397,0.7302,0.7349,0.8035,0.8025
7,9,0.8331,0.7667,0.7051,0.7346,0.8064,0.8003
15,17,0.8323,0.7639,0.7068,0.7342,0.8059,0.8002
10,12,0.8376,0.7929,0.6828,0.7338,0.8085,0.7979
17,19,0.8307,0.7626,0.7018,0.7309,0.8037,0.7976


### Validasi DTL

In [ ]:
fit_validation_model("dtl")

Melatih dtl dengan 21,857 baris training.
Selesai dtl.


### Validasi Logistic Regression


In [ ]:
fit_validation_model("logistic_regression")

Melatih logistic_regression dengan 21,857 baris training.
Selesai logistic_regression.


### Validasi SVM

In [ ]:
fit_validation_model("svm")

Melatih svm dengan 3,000 baris training.


### Validasi Naive Bayes

In [ ]:
SMOOTHING_VALUES = [1e-9, 0.001, 0.01, 0.1, 0.3, 1.0]

nb_tuning = []
for smoothing in SMOOTHING_VALUES:
    plain = naive_bayes(train_array, train_labels, smoothing=smoothing, pick_features=False)
    picked = naive_bayes(train_array, train_labels, smoothing=smoothing, pick_features=True)
    nb_tuning.append({
        "smoothing": smoothing,
        "tanpa pick_features": evaluateMetrics(plain.predict(val_array), val_labels)["macrof1Score"],
        "dengan pick_features": evaluateMetrics(picked.predict(val_array), val_labels)["macrof1Score"],
        "jumlah kolom": len(picked.used_cols),
    })
nb_tuning = pd.DataFrame(nb_tuning)
display(nb_tuning.set_index("smoothing").round(4))

In [ ]:
best_plain = nb_tuning.loc[nb_tuning["tanpa pick_features"].idxmax(), "smoothing"]
best_pick = NOTEBOOK_CONFIG["naive_bayes"]["smoothing"]

fit_validation_model("naive_bayes")
nb_plain = naive_bayes(train_array, train_labels, smoothing=best_plain, pick_features=False)
nb_sklearn = GaussianNB(var_smoothing=best_plain).fit(train_array, train_labels)

nb_results = {
    f"scikit-learn GaussianNB (var_smoothing={best_plain})": nb_sklearn.predict(val_array),
    f"Naive Bayes kita (smoothing={best_plain})": nb_plain.predict(val_array),
    f"Naive Bayes kita + pick_features (smoothing={best_pick})": predictions["naive_bayes"],
}
nb_table = pd.DataFrame({name: evaluateMetrics(pred, val_labels) for name, pred in nb_results.items()}).T
display(nb_table.round(4))
print("Kolom terpilih:", [X_train_ready.columns[i] for i in models["naive_bayes"].used_cols])

## Model Evaluation

This part compares the validation results of all models.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
validation_metrics = pd.DataFrame([
    {
        "model": name,
        "trainRows": len(training_inputs[name][1]),
        **evaluateMetrics(result, val_labels),
    }
    for name, result in predictions.items()
]).sort_values("macrof1Score", ascending=False, ignore_index=True)

display(validation_metrics.round(4))
validation_metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
# Write your code here

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

In [ ]:
#referensi yang digunakan : https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html


def getBaseline(modelName, **parameters):
    from sklearn.linear_model import SGDClassifier
    from sklearn.naive_bayes import GaussianNB
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier

    baselines = {
        "knn": (KNeighborsClassifier, {"n_neighbors": 3, "metric": "euclidean"}),
        "dtl": (DecisionTreeClassifier, {"criterion": "entropy", "random_state": 0}),
        "logistic_regression": (SGDClassifier, {
            "loss": "log_loss", "penalty": None, "learning_rate": "constant",
            "eta0": 0.01, "max_iter": 100, "tol": None, "random_state": 42,
        }),
        "svm": (SVC, {"kernel": "rbf", "random_state": 42}),
        "naive_bayes": (GaussianNB, {}),
    }
    if modelName not in baselines:
        raise ValueError(f"Model acuan tidak tersedia: {modelName!r}.")
    constructor, defaults = baselines[modelName]
    options = {**defaults, **parameters}
    if modelName == "logistic_regression" and options["loss"] != "log_loss":
        raise ValueError("Acuan Logistic Regression harus menggunakan loss log_loss.")
    try:
        return constructor(**options)
    except TypeError as error:
        raise ValueError(f"Parameter model acuan {modelName!r} tidak sesuai.") from error


get_baseline = getBaseline

In [ ]:
def baselineOptions(modelName, parameters):
    options = dict(parameters)
    if modelName == "knn":
        return {"n_neighbors": options.get("k", 3)}
    if modelName == "dtl":
        return options
    if modelName == "logistic_regression":
        regularization = options.get("regularization") or "none"
        strength = options.get("regularizationStrength", 0.01)
        return {
            "penalty": None if regularization.lower() == "none" or strength == 0 else regularization.lower(),
            "alpha": strength,
            "eta0": options.get("learningRate", 0.01),
            "max_iter": options.get("epochs", 100),
            "l1_ratio": options.get("l1Ratio", 0.5),
            "shuffle": options.get("shuffle", True),
            "random_state": options.get("randomState", 42),
        }
    if modelName == "svm":
        options.pop("max_passes", None)
        return options
    return {"var_smoothing": options.get("smoothing", 1e-9)}

In [ ]:
comparison_rows = []
for name, result in predictions.items():
    features, labels = training_inputs[name]
    options = baselineOptions(name, model_parameters[name])
    baseline = getBaseline(name, **options).fit(features, labels)
    if name == "logistic_regression":
        positive = int(np.flatnonzero(baseline.classes_ == 1)[0])
        baseline_result = (baseline.predict_proba(val_array)[:, positive] >= 0.5).astype(int)
    else:
        baseline_result = baseline.predict(val_array)
    scratch = evaluateMetrics(result, val_labels)
    reference = evaluateMetrics(baseline_result, val_labels)
    for metric, scratch_score in scratch.items():
        comparison_rows.append({
            "model": name,
            "baseline": type(baseline).__name__,
            "trainRows": len(labels),
            "metric": metric,
            "fromScratch": scratch_score,
            "scikitLearn": reference[metric],
            "difference": scratch_score - reference[metric],
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison.loc[comparison["metric"].isin(["accuracy", "f1Score", "macrof1Score"])].round(4))
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)

---

# Saving and Loading Models <a name="9"></a>

---

<!-- artifacts: <model>_v3.pkl -->

This section stores the trained models and restores them for prediction.

In [ ]:
def saveModel(model, path: str | Path) -> Path:
    path = Path(path)
    try:
        content = pickle.dumps(model, protocol=pickle.HIGHEST_PROTOCOL)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
    except (OSError, pickle.PickleError, TypeError, AttributeError) as error:
        raise ValueError(f"Model tidak dapat disimpan ke {path}.") from error
    return path


def loadModel(path: str | Path):
    path = Path(path)
    try:
        with path.open("rb") as source:
            return pickle.load(source)
    except FileNotFoundError as error:
        raise FileNotFoundError(f"Berkas model tidak ditemukan: {path}.") from error
    except (OSError, pickle.PickleError, EOFError, AttributeError, ImportError, ValueError) as error:
        raise ValueError(f"Model tidak dapat dibaca dari {path}.") from error

In [ ]:
model_dir = OUTPUT_DIR / "models"
prep_path = saveModel(prep, model_dir / "preprocessing.pkl")
loaded_prep = loadModel(prep_path)
restored_sample, _, _ = runPreprocessing(
    X_val.iloc[:8], state=loaded_prep, name="validation_reload"
)
restored_sample = restored_sample.to_numpy(dtype=float)
np.testing.assert_allclose(restored_sample, val_array[:len(restored_sample)])

for name, fitted_model in models.items():
    model_path = saveModel(fitted_model, model_dir / f"{name}.pkl")
    restored = loadModel(model_path)
    restored_predictions = notebook_predict(name, restored, restored_sample)
    np.testing.assert_array_equal(restored_predictions, predictions[name][:len(restored_sample)])

print(f"Preprocessing dan {len(models)} model berhasil dimuat ulang dari {model_dir}.")

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
if SUBMISSION_MODEL == "knn":
    submission_parameters = model_parameters["knn"]
else:
    submission_parameters = NOTEBOOK_CONFIG[SUBMISSION_MODEL]

X_full = train_df.drop(columns=[ID_COL, TARGET_COL])
y_full = train_df[TARGET_COL]
keep = ~X_full.duplicated()
X_full, y_full = X_full.loc[keep], y_full.loc[keep]
X_full_ready, y_full_ready, final_prep = runPreprocessing(
    X_full,
    y_full,
    fit=True,
    featureCount=FEATURE_COUNT,
    components=PCA_COMPONENTS,
    oversample=OVERSAMPLE,
    randomState=SEED,
    name="train_full",
)
X_test_ready, _, _ = runPreprocessing(
    test_df.drop(columns=[ID_COL]), state=final_prep, name="test"
)

submit_features = X_full_ready.to_numpy(dtype=float)
submit_labels = y_full_ready.to_numpy(dtype=int)
print(f"Fit ulang {SUBMISSION_MODEL} dengan {len(submit_labels):,} baris.", flush=True)
submission_model = TRAINERS[SUBMISSION_MODEL](
    submit_features, submit_labels, **submission_parameters
)
submit_predictions = notebook_predict(
    SUBMISSION_MODEL, submission_model, X_test_ready.to_numpy(dtype=float)
)

test_ids = pd.Index(test_df[ID_COL].astype(str))
template_ids = pd.Index(template[ID_COL].astype(str))
if not test_ids.is_unique or not template_ids.is_unique:
    raise ValueError("ID test dan sample submission harus unik.")
if len(test_ids) != len(template_ids) or set(test_ids) != set(template_ids):
    raise ValueError("ID test tidak sama dengan daftar ID sample submission.")
if len(submit_predictions) != len(test_ids) or not np.isin(submit_predictions, [0, 1]).all():
    raise ValueError("Prediksi submisi harus memuat satu label biner untuk setiap ID test.")

submission = template[[ID_COL, TARGET_COL]].copy()
lookup = pd.Series(submit_predictions, index=test_ids)
submission[TARGET_COL] = lookup.reindex(template_ids).to_numpy(dtype=int)
submission_path = OUTPUT_DIR / f"{SUBMISSION_MODEL}.csv"
submission.to_csv(submission_path, index=False, lineterminator="\n")
saveModel(
    {"model": submission_model, "preprocessing": final_prep},
    OUTPUT_DIR / "models" / f"{SUBMISSION_MODEL}_final.pkl",
)
print(f"{len(submission):,} prediksi tersimpan di {submission_path}.")
display(submission.head())

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.

In [ ]:
def predict_cost_sensitive(p_cancel, *, fp_cost=2, fn_cost=1):
    threshold = fp_cost / (fp_cost + fn_cost)
    return (np.asarray(p_cancel, dtype=float) >= threshold).astype(int)


def evaluate_cost(y_true, y_pred, *, fp_cost=2, fn_cost=1):
    _, fp, fn, _ = confusionCounts(y_pred, y_true)
    return {
        "FP": fp,
        "FN": fn,
        "F1 binary": evaluateMetrics(y_pred, y_true)["f1Score"],
        "Total biaya": fp_cost * fp + fn_cost * fn,
    }

confusion_rows = []
for name, result in predictions.items():
    tn, fp, fn, tp = confusionCounts(result, val_labels)
    confusion_rows.append({"model": name, "TN": tn, "FP": fp, "FN": fn, "TP": tp})
error_summary = pd.DataFrame(confusion_rows)
display(error_summary)
error_summary.to_csv(OUTPUT_DIR / "confusion_counts.csv", index=False)

p_cancel = knn_neighbors[:, :best_k].mean(axis=1)
cost_sensitive_prediction = predict_cost_sensitive(p_cancel, fp_cost=2, fn_cost=1)
cost_comparison = pd.DataFrame([
    {"aturan": "Voting mayoritas", **evaluate_cost(val_labels, predictions["knn"])},
    {"aturan": "Cost-sensitive (2/3)", **evaluate_cost(val_labels, cost_sensitive_prediction)},
])
display(cost_comparison.round(4))
cost_comparison.to_csv(OUTPUT_DIR / "knn_cost_comparison.csv", index=False)

### Analisis Cost-Sensitive KNN

False positive (FP) terjadi ketika model memprediksi pemesanan akan batal,
padahal pemesanan masih aktif. Kesalahan ini bisa membuat hotel mengira
kamar akan kosong dan menerima pemesanan tambahan. Karena itu, kami
memberi bobot biaya 2 untuk FP dan 1 untuk false negative (FN).

Pada KNN dengan k=5, voting mayoritas membutuhkan 3 suara untuk
memprediksi batal. Aturan cost-sensitive menaikkan ambang menjadi 2/3,
sehingga prediksi batal membutuhkan minimal 4 dari 5 suara.

Pada 5.475 baris validasi, FP turun dari 408 menjadi 179, sedangkan FN
naik dari 494 menjadi 757. Berkurangnya 229 FP menghemat 458 unit biaya.
Tambahan 263 FN menambah 263 unit biaya. Hasil akhirnya, total biaya
turun dari 1.310 menjadi 1.115, atau sekitar 14,9%.

F1 binary turun dari 0,7424 menjadi 0,6890 karena lebih banyak pembatalan
aktual terlewat. Dalam skenario ini, aturan cost-sensitive memenuhi
tujuannya karena penghematan dari penurunan FP lebih besar daripada
biaya tambahan akibat kenaikan FN.